# Customer Feature Engineering

## 1. Objective

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from urllib.parse import quote_plus
from sqlalchemy import create_engine
from pathlib import Path
from sklearn.preprocessing import StandardScaler

## 2. Data Loading

Load the customer-level behavioral dataset created in SQL Server for feature engineering.

In [ ]:
server = os.getenv("SQL_SERVER", "localhost")
database = "EcommerceAnalysis"

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

In [ ]:
encoded_connection_string = quote_plus(connection_string)

In [ ]:
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={encoded_connection_string}"
)

In [ ]:
sql_path = Path("..") / "01_data_preparation" / "01_customer_dataset.sql"

sql_path

In [ ]:
customer_query = sql_path.read_text(encoding="utf-8")

In [ ]:
df = pd.read_sql(customer_query, engine)

## 3. Feature Selection

Select the behavioral features that will be evaluated as inputs for customer segmentation.

Customer identifiers and profiling variables are kept separately and are not used directly as clustering inputs.

In [ ]:
candidate_features = [
    "recency_days",
    "frequency",
    "monetary",
    "avg_order_value",
    "avg_items_per_order",
    "category_diversity",
    "favorite_category_share",
    "purchase_frequency",
    "effective_discount_rate",
    "returned_order_rate"
]

In [ ]:
X = df[candidate_features].copy()

## 4. Missing Value Check

In [ ]:
X.isna().sum()

## 5. Skewness Transformation

Apply log transformation to selected right-skewed features to reduce the influence
of extreme values and create more balanced distributions before clustering.

In [ ]:
log_features = [
    "frequency",
    "monetary",
    "avg_order_value",
    "purchase_frequency"
]

In [ ]:
for feature in log_features:
    X[f"{feature}_log"] = np.log1p(X[feature])

In [ ]:
for feature in log_features:
    original_skew = X[feature].skew()
    log_skew = X[f"{feature}_log"].skew()

    print(
        f"{feature}: "
        f"original = {original_skew:.2f}, "
        f"log = {log_skew:.2f}"
    )

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(X["purchase_frequency"], bins=30)
plt.title("Purchase Frequency - Original")
plt.xlabel("Purchase Frequency")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(X["purchase_frequency_log"], bins=30)
plt.title("Purchase Frequency - Log Transformed")
plt.xlabel("Log Purchase Frequency")
plt.ylabel("Frequency")
plt.show()

## 6. Model Feature Set

Create a model-specific feature set using the transformed versions of highly
skewed features while retaining the remaining behavioral features.

In [ ]:
X_model = X[
    [
        "recency_days",
        "frequency_log",
        "monetary_log",
        "avg_order_value_log",
        "avg_items_per_order",
        "category_diversity",
        "favorite_category_share",
        "purchase_frequency_log",
        "effective_discount_rate",
        "returned_order_rate"
    ]
].copy()

In [ ]:
X_model.agg(["min", "max"])

## 7. Feature Scaling

Standardize the model features before K-Means clustering so that variables with
larger numerical scales do not dominate the distance calculations.

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_model)

In [ ]:
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=X_model.columns,
    index=X_model.index
)

In [ ]:
X_scaled_df.agg(["mean", "std"])

## 8. Feature Engineering Decisions

The following preprocessing decisions were made based on the exploratory analysis:

- No missing values were found in the selected clustering features, so no imputation was required.
- `frequency`, `monetary`, `avg_order_value`, and `purchase_frequency` were log-transformed to reduce positive skewness and the influence of extreme values.
- Log transformation substantially improved the distributions of the selected features, although `purchase_frequency` remained positively skewed.
- Original and log-transformed versions of the same features were not used together in the model feature set to avoid duplicate representation of the same behavior.
- All model features were standardized using `StandardScaler` because K-Means relies on distance calculations and is sensitive to differences in numerical scale.
- `category_diversity`, `favorite_category_share`, and `returned_order_rate` were retained for further evaluation rather than removed at this stage.
- Alternative feature sets will be compared during clustering to evaluate the impact of correlated and zero-heavy features.
- `average_rating` and `review_rate` were excluded from the primary clustering inputs but remain available for cluster profiling and interpretation.